# Antigravity Image Pipeline — Colab Worker

**Instructions:**
1. Set Runtime → GPU (T4 is free)
2. Run All Cells (`Ctrl+F9`)
3. Copy the `gradio.live` URL from the last cell output
4. Paste it into the local UI or CLI `--remote-url` flag

> ⚠️ Keep this tab open. The server runs as long as the notebook is active.

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────
# Run once per session. Takes ~2-3 minutes.
!pip install -q gradio numpy pillow scipy opencv-python-headless
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q basicsr realesrgan
print('\n✓ All dependencies installed')

In [ ]:
# ── Cell 2: Define the enhancement function ────────────────────────────────
import os
import time
import numpy as np
import torch
import cv2
from PIL import Image
from urllib.request import urlretrieve

os.makedirs('models', exist_ok=True)

# Cache the upsampler so we don't reload the model on every call
_upsampler_cache = {}

def _get_upsampler(scale: int, tile: int):
    key = (scale, tile)
    if key in _upsampler_cache:
        return _upsampler_cache[key]

    from basicsr.archs.rrdbnet_arch import RRDBNet
    from realesrgan import RealESRGANer

    model_path = 'models/RealESRGAN_x4plus.pth'
    if not os.path.exists(model_path):
        print('Downloading RealESRGAN_x4plus.pth ...')
        urlretrieve(
            'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
            model_path
        )
        print('✓ Downloaded')

    num_gpus = min(torch.cuda.device_count(), 4)
    device   = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    print(f'Device: {device}  |  GPUs available: {num_gpus}')

    net = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64,
                  num_block=23, num_grow_ch=32, scale=4)
    sampler = RealESRGANer(
        scale=4,
        model_path=model_path,
        model=net,
        tile=tile if tile > 0 else 0,
        tile_pad=10,
        pre_pad=0,
        half=torch.cuda.is_available(),
        device=device,
    )
    _upsampler_cache[key] = sampler
    return sampler


def enhance(image, method: str, scale: int, tile: int, face_enhance: bool):
    """
    Main API function called by gradio_client from the local machine.

    Args:
        image       : numpy uint8 RGB array (Gradio converts the file automatically)
        method      : 'realesrgan', 'bicubic', or 'lanczos'
        scale       : upscale factor (2, 4, or 8)
        tile        : tile size for VRAM management (0 = auto)
        face_enhance: whether to run GFPGAN face restoration (not supported here)

    Returns:
        Enhanced image as numpy uint8 RGB array.
    """
    scale = int(scale)
    tile  = int(tile)
    method = str(method).strip().lower()

    print(f'[Colab Worker] method={method}  scale={scale}  tile={tile}  shape={image.shape}')
    t0 = time.time()

    if method == 'realesrgan':
        upsampler = _get_upsampler(scale, tile)
        bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        out_bgr, _ = upsampler.enhance(bgr, outscale=scale)
        result = cv2.cvtColor(out_bgr, cv2.COLOR_BGR2RGB)

    elif method in ('bicubic', 'lanczos'):
        pil_img  = Image.fromarray(image)
        resample = Image.BICUBIC if method == 'bicubic' else Image.LANCZOS
        w, h     = pil_img.size
        result   = np.array(pil_img.resize((w * scale, h * scale), resample))

    else:
        raise ValueError(f'Unknown method: {method!r}')

    print(f'[Colab Worker] Done in {time.time()-t0:.1f}s  output={result.shape}')
    return result


print('✓ enhance() function defined')

In [ ]:
# ── Cell 3: Launch Gradio API server ──────────────────────────────────────
# This cell stays running. Keep this tab open!
import gradio as gr

with gr.Blocks(title='Antigravity Colab Worker') as demo:
    gr.Markdown('## Antigravity Colab Worker\nLeave this tab open while using the local UI.')

    with gr.Row():
        img_in   = gr.Image(type='numpy', label='Input Image')
        img_out  = gr.Image(type='numpy', label='Enhanced Output')

    method_in  = gr.Textbox(value='realesrgan', label='Method')
    scale_in   = gr.Number(value=4,   label='Scale',        precision=0)
    tile_in    = gr.Number(value=0,   label='Tile Size',    precision=0)
    face_in    = gr.Checkbox(value=False, label='Face Enhance')

    run_btn = gr.Button('Run (for manual testing)')
    run_btn.click(
        fn=enhance,
        inputs=[img_in, method_in, scale_in, tile_in, face_in],
        outputs=img_out,
        api_name='enhance',   # ← this is what gradio_client calls
    )

# queue() is REQUIRED for API calls to work in Gradio 4.x+
demo.queue(max_size=4)

print('Starting server ...')
demo.launch(
    share=True,       # generates the public gradio.live URL
    debug=False,
    show_error=True,
    quiet=False,
    inline=False,     # don't show the iframe inside Colab
)

# Keep the cell alive so the server doesn't shut down
print('\n✓ Server is running. Copy the URL above and paste into your local CLI/UI.')
print('  Leave this tab open!')